In [1]:
import torch
import torchvision
import os
from os.path import join as j_
from PIL import Image
import pandas as pd
import numpy as np

# loading all packages here to start
from uni import get_encoder
from uni.downstream.extract_patch_features import extract_patch_features_from_dataloader
from uni.downstream.eval_patch_features.linear_probe import eval_linear_probe
from uni.downstream.eval_patch_features.fewshot import eval_knn, eval_fewshot
from uni.downstream.eval_patch_features.protonet import ProtoNet, prototype_topk_vote
from uni.downstream.eval_patch_features.metrics import get_eval_metrics, print_metrics
from uni.downstream.utils import concat_images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FAISS import failed: ModuleNotFoundError("No module named 'faiss'")


In [42]:
class CustomImageDataset(Dataset):
    def __init__(self, df, label_col="pT"):
        self.items = []
        skipped = 0
        for _, row in df.iterrows():
            path = row["patch_path"]
            coord_path = path.replace("_patches.npy", "_coords.npy")
            coords = np.load(coord_path, allow_pickle=True)

            bag = np.load(path, allow_pickle=True)
            for i in range(bag.shape[0]):
                self.items.append((path, i, row[label_col], row["Clinical_ID"], coords[i]))
        
        print(f"Kept {len(self.items)} patches, skipped {skipped} blank patches")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, patch_idx, label, patient_id, coord = self.items[idx]
        bag = np.load(path, allow_pickle=True)
        patch = bag[patch_idx]
        return patch, label, patient_id, path, coord

def collate_fun(batch):
    imgs, labels, patient_ids, paths, coords = zip(*batch)
    imgs = torch.stack([
        torch.from_numpy(img).permute(2, 0, 1).float()
        for img in imgs
    ])
    labels = torch.tensor(labels, dtype=torch.long)
    coords = np.array(coords)
    return imgs, labels, list(patient_ids), list(paths), coords
    
####
dataset = CustomImageDataset(df_clean, label_col="pT")
dataset_loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=12, collate_fn=collate_fun)

device = next(model.parameters()).device
dataset_features = extract_patch_features_from_dataloader(model, dataset_loader, device)
dataset_feats  = torch.tensor(dataset_features["embeddings"], dtype=torch.float32)
dataset_labels = torch.tensor(dataset_features["labels"],     dtype=torch.long)
dataset_pids   = dataset_features["patient_ids"]
dataset_paths  = dataset_features["paths"]
dataset_coords = dataset_features["coords"]


Kept 84442 patches, skipped 0 blank patches


100%|█████████████████████████████████████| 21111/21111 [46:35<00:00,  7.55it/s]


In [43]:
# Are any rows truly identical?
feats = dataset_feats  # your tensor

# Check exact duplicates
n_unique = len(torch.unique(feats, dim=0))
print(f"Total rows: {len(feats)}, Unique rows: {n_unique}")

# Check pairwise similarity within patient 0
p0_mask = dataset_pids == 0
p0_feats = feats[p0_mask]
sim = torch.nn.functional.cosine_similarity(p0_feats[0].unsqueeze(0), p0_feats, dim=1)
print(f"\nPatient 0 cosine similarity to patch 0:")
print(f"  mean: {sim.mean():.4f}, min: {sim.min():.4f}, max: {sim.max():.4f}")
# Truly identical = 1.0000, similar tissue = 0.85-0.99, diverse = <0.85

Total rows: 84442, Unique rows: 84442

Patient 0 cosine similarity to patch 0:
  mean: 0.4254, min: 0.0518, max: 1.0000


In [44]:
df = pd.DataFrame(dataset_features["embeddings"])
df["label"] = dataset_labels
df["patient_id"] = dataset_pids
df["paths"] = dataset_paths
df["coord_x"] = dataset_coords[:, 0]
df["coord_y"] = dataset_coords[:, 1]